In [1]:
!pip install groq -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 13.9 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from groq import Groq

# Load API key from Colab Secrets
api_key = userdata.get("GROQ_API_KEY")

# Initialize Groq client
client = Groq(api_key=api_key)

# Model we'll use throughout the notebook
MODEL = "llama-3.3-70b-versatile"

print("Groq client ready.")
print(f"Model: {MODEL}")

Groq client ready.
Model: llama-3.3-70b-versatile


In [3]:
def call_groq(prompt, system_prompt=None, temperature=0.7, max_tokens=1024):
    """
    Simple wrapper around Groq chat completion.

    Args:
        prompt        : The user message
        system_prompt : Optional system-level instruction
        temperature   : 0 = deterministic, 1 = creative
        max_tokens    : Max length of response

    Returns:
        response text as string
    """
    messages = []

    if system_prompt:
        messages.append({"role": "system", "content": system_prompt})

    messages.append({"role": "user", "content": prompt})

    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=temperature,
        max_tokens=max_tokens
    )

    return response.choices[0].message.content

# Quick sanity test
result = call_groq("What is 2 + 2? Answer in one word.")
print("Test response:", result)

Test response: Four.


In [7]:
# Cell 3: Chain of Thought (Zero-Shot vs Direct)

problem = """
A store sells apples for Rs. 10 each and oranges for Rs. 15 each.
Vedant buys some apples and oranges. He pays with a Rs. 200 note
and gets Rs. 40 back as change. The number of oranges he bought
is twice the number of apples. How many of each did he buy?

"""

direct_prompt = f"{problem}\nGive the final answer only. No explanation."

cot_prompt = f"{problem}\nThink step by step. Show all calculations clearly."

print("WITHOUT CoT:")
print("-" * 40)
print(call_groq(direct_prompt, temperature=0.2))

print("\nWITH CoT:")
print("-" * 40)
print(call_groq(cot_prompt, temperature=0.2))

WITHOUT CoT:
----------------------------------------
Apples: 4, Oranges: 8

WITH CoT:
----------------------------------------
To solve this problem, let's break it down step by step.

1. **Understanding the transaction**: Vedant pays with a Rs. 200 note and gets Rs. 40 back as change. This means the total cost of the items he bought is Rs. 200 - Rs. 40 = Rs. 160.

2. **Setting up the equation**: Let's denote the number of apples as A and the number of oranges as O. We know:
   - The price of one apple is Rs. 10.
   - The price of one orange is Rs. 15.
   - The total cost of the items is Rs. 160.
   - The number of oranges is twice the number of apples, so O = 2A.

3. **Creating the equation based on the total cost**: The total cost can be represented by the equation 10A + 15O = 160. Since O = 2A, we can substitute O in the equation:
   - 10A + 15(2A) = 160
   - 10A + 30A = 160
   - 40A = 160

4. **Solving for A (the number of apples)**: Divide both sides of the equation by 40 to solv

In [8]:
# Cell 4: Chain of Thought - Few Shot (Clean)

few_shot_prompt = """
Solve word problems step by step like this example:

EXAMPLE:
Problem: Pens cost Rs. 5, notebooks cost Rs. 20.
Total spent = Rs. 70. Pens bought = twice the notebooks. How many each?

Solution:
Step 1: Let notebooks = x, pens = 2x
Step 2: 5(2x) + 20(x) = 70 → 10x + 20x = 30x = 70 → x = 2.33 (not whole)
        Adjust: notebooks=2, pens=4 → 5(4)+20(2) = 20+40 = 60
        notebooks=3, pens=6 → 5(6)+20(3) = 30+60 = 90
        notebooks=2, pens=4, spent=60 is valid.
Step 3: Answer: 4 pens, 2 notebooks.

---

NOW YOUR TURN:

Problem: Chai costs Rs. 10, samosa costs Rs. 8.
Vedant pays Rs. 100 and gets Rs. 22 change.
Samosas bought = twice the chai cups. How many of each?

Note: Spent = 78. Let chai=x, samosa=2x → 10x+16x=26x=78 → x=3.
Answer should be: 3 chai, 6 samosas.

Solve it step by step.
"""

print("FEW-SHOT CoT OUTPUT:")
print("-" * 40)
print(call_groq(few_shot_prompt, temperature=0.2))

FEW-SHOT CoT OUTPUT:
----------------------------------------
Problem: Chai costs Rs. 10, samosa costs Rs. 8. 
Vedant pays Rs. 100 and gets Rs. 22 change. 
Samosas bought = twice the chai cups. How many of each?

Step 1: Let chai = x, samosas = 2x
Since samosas bought are twice the chai cups, we can represent the number of samosas as 2x.

Step 2: Total spent = Rs. 100 - Rs. 22 = Rs. 78
The total amount spent is the sum of the cost of chai and samosas. 
So, 10x + 8(2x) = 78 
→ 10x + 16x = 26x = 78 
→ x = 78 / 26 = 3

Step 3: Answer: Since x = 3, the number of chai cups is 3 and the number of samosas is 2x = 2 * 3 = 6.
Therefore, the answer is: 3 chai, 6 samosas.


In [9]:
# Cell 5: ReAct - Reason + Act Pattern

# ReAct = model reasons about WHAT to do, then we execute the action,
# feed result back, model reasons again, repeat until done.
# We simulate tools manually here - no LangChain needed.

# --- Define our fake tools ---

def search_tool(query):
    """Simulates a web search. Returns fake but realistic data."""
    database = {
        "python salary india 2024": "Average Python developer salary in India is Rs. 6-12 LPA for freshers, Rs. 12-25 LPA for mid-level.",
        "machine learning engineer salary india 2024": "ML Engineer salary in India ranges from Rs. 8-15 LPA entry level, Rs. 20-40 LPA for senior roles.",
        "data scientist salary india 2024": "Data Scientist salary in India: Rs. 6-10 LPA fresher, Rs. 15-30 LPA experienced.",
        "top ml skills 2024": "Top ML skills: Python, PyTorch, Hugging Face, MLOps, LLMs, RAG, Vector Databases, Docker.",
    }
    query = query.lower()
    for key in database:
        if any(word in query for word in key.split()):
            return database[key]
    return "No results found for: " + query

def calculator_tool(expression):
    """Safely evaluates a math expression."""
    try:
        result = eval(expression, {"__builtins__": {}})
        return f"Result: {result}"
    except:
        return "Could not calculate: " + expression

tools = {
    "search": search_tool,
    "calculate": calculator_tool
}

print("Tools ready:", list(tools.keys()))

Tools ready: ['search', 'calculate']


In [10]:
# Cell 5b: ReAct Loop

import re

def react_loop(question, max_steps=5):
    """
    Runs the ReAct loop:
    Thought → Action → Observation → Thought → ... → Final Answer
    """

    system_prompt = """You are a reasoning agent. You solve problems step by step using tools.

You have access to these tools:
- search(query)     : Search for information
- calculate(expression) : Do math calculations

Always follow this EXACT format for each step:

Thought: <your reasoning about what to do next>
Action: <tool_name>(<input>)
Observation: <you will be given the result here>

When you have enough information, finish with:
Final Answer: <your complete answer>

Important: Only write ONE Thought and ONE Action per response. Stop after the Action line and wait."""

    messages = [{"role": "user", "content": f"Question: {question}"}]

    print(f"Question: {question}")
    print("=" * 50)

    for step in range(max_steps):
        print(f"\nStep {step + 1}:")
        print("-" * 30)

        # Get model's next thought + action
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "system", "content": system_prompt}] + messages,
            temperature=0.2,
            max_tokens=300
        )

        model_output = response.choices[0].message.content
        print(model_output)

        # Check if model reached final answer
        if "Final Answer:" in model_output:
            print("\n" + "=" * 50)
            print("REACT LOOP COMPLETE")
            break

        # Parse the action the model wants to take
        action_match = re.search(r"Action:\s*(\w+)\((.+?)\)", model_output)

        if action_match:
            tool_name = action_match.group(1).strip()
            tool_input = action_match.group(2).strip().strip('"').strip("'")

            # Execute the tool
            if tool_name in tools:
                observation = tools[tool_name](tool_input)
            else:
                observation = f"Tool '{tool_name}' not found."

            print(f"Observation: {observation}")

            # Feed result back into conversation
            messages.append({"role": "assistant", "content": model_output})
            messages.append({"role": "user", "content": f"Observation: {observation}"})

        else:
            print("No valid action found. Stopping loop.")
            break

# --- Run it ---
react_loop("I am transitioning from Mechanical Engineering to ML. What salary can I expect and what skills should I focus on?")

Question: I am transitioning from Mechanical Engineering to ML. What salary can I expect and what skills should I focus on?

Step 1:
------------------------------
Thought: To determine the expected salary and necessary skills for a transition from Mechanical Engineering to Machine Learning (ML), I should first research the average salary range for ML professionals, as this will provide a baseline for understanding the financial implications of the career change.

Action: search("average salary for machine learning professionals")
Observation: Average Python developer salary in India is Rs. 6-12 LPA for freshers, Rs. 12-25 LPA for mid-level.

Step 2:
------------------------------
Thought: Knowing the average salary range for ML professionals in India, the next step is to identify the key skills required for a career in Machine Learning, especially for someone transitioning from a Mechanical Engineering background, to understand what areas to focus on for a successful transition.

Acti

 **ReAct is not a model feature

— it's a prompting pattern + orchestration loop you build around the model. The model just follows format instructions. You control the loop.**

In [11]:
# Cell 6: Self-Consistency
# Run the same problem multiple times with high temperature
# Then majority vote on the final answer

from collections import Counter
import re

def extract_final_answer(text):
    """
    Pulls the final number/answer from model response.
    Looks for patterns like 'answer is X' or just the last number mentioned.
    """
    # Try to find explicit answer statement
    patterns = [
        r"answer is[:\s]+(\d+)",
        r"total[:\s]+(\d+)",
        r"final[:\s]+(\d+)",
        r"=\s*(\d+)\s*$",
        r"(\d+)\s*(?:apples|items|units|chai|samosas)?\.?\s*$"
    ]

    text_lower = text.lower()
    for pattern in patterns:
        match = re.search(pattern, text_lower, re.MULTILINE)
        if match:
            return match.group(1)

    # Fallback: return last number found in text
    numbers = re.findall(r'\d+', text)
    return numbers[-1] if numbers else "unknown"


def self_consistency(question, n_samples=5):
    """
    Runs the same question n times at high temperature.
    Collects all answers and returns the majority vote.
    """

    prompt = f"{question}\nThink step by step and give a clear final answer."

    print(f"Question: {question}")
    print(f"Running {n_samples} independent samples...\n")
    print("=" * 50)

    answers = []

    for i in range(n_samples):
        response = call_groq(
            prompt,
            temperature=0.8,   # HIGH temperature = different reasoning paths each time
            max_tokens=400
        )

        extracted = extract_final_answer(response)
        answers.append(extracted)

        print(f"Sample {i+1}:")
        print(response[:300] + "..." if len(response) > 300 else response)
        print(f"Extracted Answer: {extracted}")
        print("-" * 40)

    # Majority vote
    vote_counts = Counter(answers)
    majority_answer = vote_counts.most_common(1)[0][0]

    print("\n" + "=" * 50)
    print("VOTING RESULTS:")
    for answer, count in vote_counts.most_common():
        bar = "█" * count
        print(f"  Answer {answer}: {bar} ({count}/{n_samples} votes)")

    print(f"\nFINAL ANSWER (majority vote): {majority_answer}")
    return majority_answer


# --- Run it ---
# Same verified problem from Cell 3
question = """
A store sells apples for Rs. 10 each and oranges for Rs. 15 each.
Vedant pays Rs. 200 and gets Rs. 40 change.
He bought twice as many oranges as apples.
How many apples did he buy? (Correct answer is 4)
"""

result = self_consistency(question, n_samples=5)

Question: 
A store sells apples for Rs. 10 each and oranges for Rs. 15 each.
Vedant pays Rs. 200 and gets Rs. 40 change.
He bought twice as many oranges as apples.
How many apples did he buy? (Correct answer is 4)

Running 5 independent samples...

Sample 1:
To solve this problem, let's follow these steps:

1. Calculate the total amount Vedant spent on fruits.
2. Set up an equation based on the given information about the number of apples and oranges.
3. Solve the equation to find the number of apples Vedant bought.

Step 1: Calculate the total amount V...
Extracted Answer: 160
----------------------------------------
Sample 2:
To solve this problem, let's break it down step by step:

1. **Total Cost and Change**: Vedant pays Rs. 200 and gets Rs. 40 change. This means the total cost of the items he bought is Rs. 200 - Rs. 40 = Rs. 160.

2. **Relationship between Apples and Oranges**: He bought twice as many oranges as appl...
Extracted Answer: 160
--------------------------------------

**Key insights**

At temperature=0.2 — model is deterministic, same answer every time, no point running multiple times
At temperature=0.8 — model explores different paths, some correct, some wrong
Majority vote filters out the wrong paths statistically

Cost tradeoff: 5 samples = 5x the API calls. In production you'd use this only for high-stakes decisions

**What to observe:**

If all 5 agree → high confidence answer

If votes are split 3-2 → model is uncertain, problem is ambiguous

If all 5 disagree → the problem or prompt is broken

In [12]:
# Cell 7: Tree of Thought (ToT)
# Instead of one reasoning chain, explore MULTIPLE approaches
# Evaluate each branch, keep the best, go deeper on it.

# Structure:
# Problem
# ├── Approach A → score it
# ├── Approach B → score it
# └── Approach C → score it
#         └── Best approach → expand deeper → Final Answer

def generate_approaches(problem, n_approaches=3):
    """
    Step 1: Ask model to think of multiple different ways to solve the problem.
    """
    prompt = f"""
You are a problem-solving expert.

Given this problem:
{problem}

Generate exactly {n_approaches} DIFFERENT approaches to solve it.
Each approach should use a different strategy or angle.

Format your response exactly like this:
Approach 1: <describe the approach in 2-3 sentences>
Approach 2: <describe the approach in 2-3 sentences>
Approach 3: <describe the approach in 2-3 sentences>
"""
    response = call_groq(prompt, temperature=0.7, max_tokens=400)

    # Parse approaches
    approaches = []
    for i in range(1, n_approaches + 1):
        pattern = rf"Approach {i}:(.*?)(?=Approach {i+1}:|$)"
        match = re.search(pattern, response, re.DOTALL)
        if match:
            approaches.append(match.group(1).strip())

    return approaches


def evaluate_approach(problem, approach):
    """
    Step 2: Score each approach on feasibility and clarity (1-10).
    """
    prompt = f"""
Problem: {problem}

Proposed Approach: {approach}

Evaluate this approach on two criteria:
1. Feasibility: Can this actually solve the problem? (1-10)
2. Clarity: Is the reasoning path clear and logical? (1-10)

Then give a Total Score (average of both).

Respond in this exact format:
Feasibility: <score>/10
Clarity: <score>/10
Total Score: <score>/10
Reasoning: <one sentence why>
"""
    response = call_groq(prompt, temperature=0.2, max_tokens=200)

    # Extract total score
    match = re.search(r"Total Score:\s*(\d+(?:\.\d+)?)/10", response)
    score = float(match.group(1)) if match else 0.0

    return score, response


def solve_with_best_approach(problem, approach):
    """
    Step 3: Take the winning approach and fully solve the problem with it.
    """
    prompt = f"""
Problem: {problem}

Use this specific approach to solve it:
{approach}

Now execute this approach completely, step by step.
Show all your work clearly.
End with: Final Answer: <answer>
"""
    return call_groq(prompt, temperature=0.2, max_tokens=600)


def tree_of_thought(problem):
    """
    Full ToT pipeline:
    Generate branches → Evaluate → Pick best → Solve
    """
    import re

    print(f"PROBLEM:\n{problem}")
    print("=" * 50)

    # Step 1: Generate approaches (branches)
    print("\nSTEP 1: Generating approaches (branches)...")
    print("-" * 40)
    approaches = generate_approaches(problem, n_approaches=3)

    for i, approach in enumerate(approaches, 1):
        print(f"Approach {i}: {approach}\n")

    # Step 2: Evaluate each branch
    print("\nSTEP 2: Evaluating each branch...")
    print("-" * 40)

    scored_approaches = []
    for i, approach in enumerate(approaches, 1):
        score, evaluation = evaluate_approach(problem, approach)
        scored_approaches.append((score, approach, evaluation))
        print(f"Approach {i} Score: {score}/10")
        print(evaluation)
        print()

    # Step 3: Pick the best branch
    scored_approaches.sort(key=lambda x: x[0], reverse=True)
    best_score, best_approach, best_evaluation = scored_approaches[0]

    print("\nSTEP 3: Best approach selected...")
    print("-" * 40)
    print(f"Winning Score: {best_score}/10")
    print(f"Winning Approach: {best_approach}")

    # Step 4: Solve using the best approach
    print("\nSTEP 4: Solving with best approach...")
    print("-" * 40)
    final_solution = solve_with_best_approach(problem, best_approach)
    print(final_solution)

    print("\n" + "=" * 50)
    print("TREE OF THOUGHT COMPLETE")

    return final_solution


# --- Run it ---
# Use a problem that genuinely has multiple valid approaches
problem = """
Vedant is transitioning from Mechanical Engineering to ML Engineering.
He has 3 months of self-study time. He needs to decide which skills
to prioritize: Deep Learning fundamentals, MLOps and deployment,
or LLMs and prompt engineering. He wants to maximize his chances
of landing an entry-level ML Engineer job in India.
What should he focus on and in what order?
"""

tree_of_thought(problem)

PROBLEM:

Vedant is transitioning from Mechanical Engineering to ML Engineering.
He has 3 months of self-study time. He needs to decide which skills 
to prioritize: Deep Learning fundamentals, MLOps and deployment, 
or LLMs and prompt engineering. He wants to maximize his chances 
of landing an entry-level ML Engineer job in India.
What should he focus on and in what order?


STEP 1: Generating approaches (branches)...
----------------------------------------
Approach 1: The "Foundational" approach focuses on building a strong foundation in Deep Learning fundamentals, as it is a crucial aspect of ML Engineering. Vedant should allocate the first month to studying Deep Learning concepts, such as neural networks and convolutional neural networks, and then move on to MLOps and deployment in the second month. This approach will provide a solid base for him to learn more advanced topics like LLMs and prompt engineering in the third month.

Approach 2: The "Market-Driven" approach involves an

'To solve this problem using the "Foundational" approach, let\'s break it down into steps and allocate Vedant\'s 3 months of self-study time accordingly.\n\n**Step 1: Month 1 - Deep Learning Fundamentals**\nVedant should start by building a strong foundation in Deep Learning. This involves studying the basics of neural networks, including:\n- Introduction to neural networks\n- Types of neural networks (feedforward, recurrent, convolutional)\n- Activation functions\n- Backpropagation\n- Optimization techniques\n\nHe should also explore convolutional neural networks (CNNs) and their applications, as well as other Deep Learning concepts like autoencoders, generative adversarial networks (GANs), and transfer learning.\n\n**Step 2: Month 2 - MLOps and Deployment**\nAfter gaining a solid understanding of Deep Learning fundamentals, Vedant should move on to MLOps and deployment in the second month. This includes:\n- Introduction to MLOps\n- Data preprocessing and feature engineering\n- Model 

**Key insights**

ToT is expensive — this makes 4 API calls minimum (1 generate + 3 evaluate + 1 solve)

The power is in the evaluation step — the model catching its own weak reasoning before committing
Used a career decision problem deliberately — ToT shines on open-ended problems not just math

Math problems have one right answer, ToT advantage is minimal there. Strategic decisions have a wide solution space — that's where ToT earns its cost

**What to observe:**

All 3 approaches will be different strategies, not variations of the same idea
Scores will differ — model genuinely differentiates between stronger and weaker approaches

Final answer will be more structured and justified than a direct prompt would give

In [13]:
# Cell 8: Meta-Prompting
# Use the model to critique and rewrite prompts themselves.
# Three patterns:
# 1. Prompt Generator   - describe a task, model writes the best prompt
# 2. Prompt Critic      - give a bad prompt, model identifies what's wrong
# 3. Prompt Improver    - bad prompt in, better prompt out, then run it

def generate_prompt(task_description):
    """
    Pattern 1: You describe what you want to achieve.
    Model writes the optimal prompt for it.
    """
    meta_prompt = f"""
You are an expert prompt engineer.

A user wants to accomplish this task:
{task_description}

Write the single best prompt they should use to accomplish this task.
The prompt should be:
- Clear and specific
- Include any necessary context
- Specify the output format
- Avoid ambiguity

Write only the prompt itself. No explanation, no preamble.
"""
    return call_groq(meta_prompt, temperature=0.3, max_tokens=400)


def critique_prompt(bad_prompt, task_description):
    """
    Pattern 2: Give the model a weak prompt.
    It identifies exactly what's wrong with it.
    """
    meta_prompt = f"""
You are an expert prompt engineer.

The goal of this prompt is to: {task_description}

Here is the prompt being used:
\"\"\"{bad_prompt}\"\"\"

Analyze this prompt and identify:
1. What is unclear or ambiguous
2. What context is missing
3. What format issues exist
4. Overall quality score (1-10)

Be specific. Point to exact phrases that are weak.
"""
    return call_groq(meta_prompt, temperature=0.2, max_tokens=400)


def improve_prompt(bad_prompt, task_description):
    """
    Pattern 3: Full pipeline.
    Bad prompt in → critique → improved prompt out → run improved prompt → compare outputs.
    """
    # Step 1: Critique
    print("STEP 1: Critiquing original prompt...")
    print("-" * 40)
    critique = call_groq(f"""
You are an expert prompt engineer.
Task goal: {task_description}
Original prompt: \"\"\"{bad_prompt}\"\"\"

Identify the top 3 weaknesses of this prompt. Be specific.
""", temperature=0.2, max_tokens=300)
    print(critique)

    # Step 2: Rewrite
    print("\nSTEP 2: Rewriting improved prompt...")
    print("-" * 40)
    improved_prompt = call_groq(f"""
You are an expert prompt engineer.
Task goal: {task_description}
Original weak prompt: \"\"\"{bad_prompt}\"\"\"

Rewrite this as a significantly better prompt.
Output only the improved prompt. No explanation.
""", temperature=0.3, max_tokens=400)
    print(improved_prompt)

    # Step 3: Run both and compare
    print("\nSTEP 3: Running original prompt...")
    print("-" * 40)
    original_output = call_groq(bad_prompt, temperature=0.3, max_tokens=300)
    print(original_output)

    print("\nSTEP 4: Running improved prompt...")
    print("-" * 40)
    improved_output = call_groq(improved_prompt, temperature=0.3, max_tokens=300)
    print(improved_output)

    return {
        "original_prompt": bad_prompt,
        "improved_prompt": improved_prompt,
        "original_output": original_output,
        "improved_output": improved_output
    }


# ── Pattern 1: Generate a prompt from scratch ──────────────────

print("PATTERN 1: PROMPT GENERATOR")
print("=" * 50)

task = """
Generate a weekly study plan for someone transitioning from
Mechanical Engineering to ML Engineering in 3 months.
"""

generated_prompt = generate_prompt(task)
print("Generated Prompt:")
print(generated_prompt)

print("\nRunning generated prompt...")
print("-" * 40)
print(call_groq(generated_prompt, temperature=0.3, max_tokens=500))


# ── Pattern 2: Critique a weak prompt ─────────────────────────

print("\n\nPATTERN 2: PROMPT CRITIC")
print("=" * 50)

weak_prompt = "Tell me about ML."

task_for_weak = "Help a mechanical engineer understand which ML topics to learn first"

print(f"Weak Prompt: '{weak_prompt}'")
print("\nCritique:")
print("-" * 40)
print(critique_prompt(weak_prompt, task_for_weak))


# ── Pattern 3: Full improve pipeline ──────────────────────────

print("\n\nPATTERN 3: FULL IMPROVE PIPELINE")
print("=" * 50)

bad_prompt = "Write something about how to get a job in AI."

task_goal = """
Give a specific, actionable job search strategy for an
entry-level ML Engineer candidate with a non-CS background
"""

results = improve_prompt(bad_prompt, task_goal)

PATTERN 1: PROMPT GENERATOR
Generated Prompt:
Create a weekly study plan for the next 3 months to transition from Mechanical Engineering to ML Engineering, assuming 20 hours of study per week, including specific topics, resources, and goals for each week, and format the plan as a table with columns for week number, topic, resource, study hours, and goals.

Running generated prompt...
----------------------------------------
To transition from Mechanical Engineering to ML Engineering, you'll need to focus on building a strong foundation in programming, mathematics, and machine learning concepts. Here's a 12-week study plan to help you achieve this goal:

| Week # | Topic | Resource | Study Hours | Goals |
| --- | --- | --- | --- | --- |
| 1 | Python Basics | Codecademy's Python Course, Python Crash Course by Eric Matthes | 20 | Learn basic syntax, data types, control structures, functions, and modules |
| 2 | Python Data Structures and File Handling | Python Crash Course by Eric Matthes

**Key insight to write down**

Meta-prompting is the technique you'll use most in real ML engineering work — not for model tasks but for building prompts for your own pipelines and products

The comparison in Pattern 3 is the real learning — the output quality difference between a bad and improved prompt is usually dramatic

This is also how you'd build a prompt optimization system — automate this loop, score outputs, keep iterating

**What to observe:**

Pattern 1 output will be significantly more structured than what you'd write manually

Pattern 2 critique will be harsh and specific — pay attention to what it flags as weak

Pattern 3 comparison: original output will be generic, improved output will be specific and actionable

 Day 2 Summary - Advanced Prompting Techniques

summary = """
# Day 2: Advanced Prompting Techniques
## Summary & Key Observations

---

## 1. Chain of Thought (CoT)

**What it does:** Forces the model to reason step by step before answering.

**Two variants:**
- Zero-shot CoT  : Just add "Think step by step" — no examples needed
- Few-shot CoT   : Give a worked example first, then ask your question

**When to use:**
- Multi-step math or logic problems
- Any task where intermediate steps matter
- Debugging, root cause analysis

**Limitation:**
- Does not fix arithmetic errors — model can reason correctly but calculate wrong
- A wrong early step confidently leads to a wrong final answer

---

## 2. ReAct (Reason + Act)

**What it does:** Model alternates between reasoning and taking actions (tool calls).
Loop: Thought → Action → Observation → Thought → ... → Final Answer

**When to use:**
- Tasks requiring external tools (search, calculator, database)
- Multi-step research or data gathering
- Building AI agents

**Limitation:**
- Expensive — each loop step is a separate API call
- Can get stuck if it misreads an observation
- Requires you to build the orchestration loop yourself

---

## 3. Self-Consistency

**What it does:** Runs the same prompt N times at high temperature, takes majority vote.

**When to use:**
- High-stakes answers where one wrong run is costly
- Classification and factual questions
- When you can afford multiple API calls

**Limitation:**
- N samples = N times the cost
- Useless if the model is systematically biased in one direction
- Does not help with open-ended creative tasks (no "majority" exists there)

---

## 4. Tree of Thought (ToT)

**What it does:** Explores multiple reasoning branches, scores each, prunes weak ones,
solves using the best branch only.

**When to use:**
- Strategic planning and decision making
- Problems with a wide solution space
- Architecture design, career decisions, complex tradeoffs

**Limitation:**
- Most expensive technique — minimum 4-5 API calls per problem
- Overkill for simple or single-answer problems
- Branch evaluation is still done by the model — it can misjudge

---

## 5. Meta-Prompting

**What it does:** Uses the model to generate, critique, and improve prompts themselves.

**Three patterns:**
- Generator : Describe task → model writes best prompt
- Critic    : Give weak prompt → model identifies exact flaws
- Improver  : Bad prompt in → critique → rewrite → compare outputs

**When to use:**
- When you do not know how to frame a new task
- Auditing existing prompts in a production pipeline
- Building prompt optimization systems

**Limitation:**
- Model's meta-level reasoning can also be wrong — outputs need testing
- Can create bloated prompts if you stack too many layers
- Not a substitute for domain knowledge in prompt design

---

## Technique Selection Guide

| Technique        | Best For                        | API Calls | Use In Production? |
|------------------|---------------------------------|-----------|-------------------|
| CoT Zero-shot    | Quick reasoning tasks           | 1         | Yes               |
| CoT Few-shot     | Consistent reasoning format     | 1         | Yes               |
| ReAct            | Tool use and agents             | N (loop)  | Yes               |
| Self-Consistency | High-stakes single answers      | N         | Selective         |
| Tree of Thought  | Strategic / open-ended problems | 4+        | Selective         |
| Meta-Prompting   | Building and auditing prompts   | 2-4       | Yes               |

---

## Biggest Takeaway

These are not model features. They are prompting patterns and orchestration logic
you build around the model. The model just follows instructions.
Your job as an ML Engineer is to know which pattern fits which problem
and implement the loop correctly around it.
"""

print(summary)